# Chatbot evaluator grounding specialist

This notebook prepares public Hugging Face grounding data and trains a QLoRA adapter. It does **not** replace the evaluator's deterministic citation checks or human-review queue.


In [1]:
import importlib.util
from pathlib import Path
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('Dependencies:', {m: importlib.util.find_spec(m) is not None for m in ['transformers', 'datasets', 'peft', 'bitsandbytes', 'accelerate']})
print('Uploaded script:', Path('/content/colab_train.py').exists())
print('Repo:', Path('/content/chatbot-evaluator').exists())
print('Data:', Path('/content/chatbot-evaluator/training/ragtruth-grounding.jsonl').exists())

GPU: Tesla T4
Dependencies: {'transformers': True, 'datasets': True, 'peft': True, 'bitsandbytes': False, 'accelerate': True}
Uploaded script: True
Repo: False
Data: False


In [2]:
%pip -q install bitsandbytes
%pip -q uninstall -y torchao
!git clone https://github.com/Kapilpaliwal42/chatbot-evaluator.git /content/chatbot-evaluator
%cd /content/chatbot-evaluator
!git log -1 --oneline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.1 MB/s eta 0:00:00
Cloning into '/content/chatbot-evaluator'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 109 (delta 35), reused 96 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 151.57 KiB | 1.21 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/chatbot-evaluator
bd9dd87 (HEAD -> main, origin/main, origin/HEAD) Use quantized standard-loss trainer on T4


In [3]:
!python training/prepare_hf_data.py --dataset wandb/RAGTruth-processed --out training/ragtruth-grounding.jsonl --limit 10000
!cat training/ragtruth-grounding.manifest.json

README.md: 100% 1.47k/1.47k [00:00<00:00, 4.93MB/s]
Repo card metadata block was not found. Setting CardData to empty.

data/train-00000-of-00001.parquet: downloading bytes:  65% 14.4M/22.3M [00:00<00:00, 24.6MB/s,  413kB/s  ]
data/train-00000-of-00001.parquet: downloading bytes: 100% 20.6M/20.6M [00:00<00:00, 27.0MB/s, 2.01MB/s  ]
data/train-00000-of-00001.parquet: reconstructing file: 100% 22.3M/22.3M [00:00<00:00, 29.2MB/s, 2.19MB/s  ]

data/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/3.88M [00:00<?, ?B/s]
data/test-00000-of-00001.parquet: downloading bytes: 100% 3.58M/3.58M [00:00<00:00, 6.91MB/s,  354kB/s  ]
data/test-00000-of-00001.parquet: reconstructing file: 100% 3.88M/3.88M [00:00<00:00, 7.48MB/s,  384kB/s  ]
Generating train split: 100% 15090/15090 [00:00<00:00, 64922.78 examples/s]
Generating test split: 100% 2700/2700 [00:00<00:00, 41674.93 examples/s]
{
  "dataset": "wandb/RAGTruth-processed",
  "split": "train",
  "rows": 10000,
  "skipped": 0,
  "labels": 

In [4]:
!python /content/colab_train.py --data /content/chatbot-evaluator/training/ragtruth-grounding.jsonl --model Qwen/Qwen2.5-Coder-7B-Instruct --out /content/grounding-smoke --max-steps 2 --gradient-accumulation-steps 1 --batch-size 1 --max-length 1536

config.json: 100% 663/663 [00:00<00:00, 2.25MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 21.2MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 65.1MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 123MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 168MB/s]
{
  "input_rows": 10000,
  "retained_rows": 9275,
  "overlength_skipped": 725,
  "label_counts": {
    "supported": 4464,
    "unsupported": 4811
  }
}
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 73.6MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/5.42G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/15.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/15.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/15.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   1% 90.3M/15.2G [00:02<07:45, 32.5MB/s, 26.1MB/s  

In [5]:
from pathlib import Path
import json
from safetensors.torch import load_file
p = Path('/content/grounding-smoke')
m = json.loads((p / 'run_metadata.json').read_text())
w = load_file(str(p / 'adapter_model.safetensors'))
b = [v for k, v in w.items() if 'lora_B' in k]
assert b and any(v.count_nonzero().item() > 0 for v in b)
print('PASS: two optimizer steps completed and saved LoRA B weights are nonzero.')
print('Trainable parameter dtypes:', m['trainable_dtypes'])
print('Data:', m['data'])
print('Dependency versions:', m['versions'])
print('Saved files:', {f.name: f.stat().st_size for f in p.iterdir() if f.is_file()})

PASS: two optimizer steps completed and saved LoRA B weights are nonzero.
Trainable parameter dtypes: {'torch.float32': 10092544}
Data: {'input_rows': 10000, 'retained_rows': 9275, 'overlength_skipped': 725, 'label_counts': {'supported': 4464, 'unsupported': 4811}}
Dependency versions: {'torch': '2.11.0+cu130', 'transformers': '5.18.0', 'peft': '0.21.1', 'accelerate': '1.15.0', 'bitsandbytes': '0.50.2', 'datasets': '4.8.5'}
Saved files: {'tokenizer.json': 11421892, 'README.md': 5214, 'run_metadata.json': 488, 'adapter_model.safetensors': 40400200, 'adapter_config.json': 1137, 'tokenizer_config.json': 693, 'training_args.bin': 5201, 'chat_template.jinja': 2507}


In [ ]:
!python -u /content/run_grounding_job.py --train-data /content/chatbot-evaluator/training/ragtruth-grounding.jsonl --out /content/grounding-full --epochs 2

{"phase": "freeze unseen public evaluation", "state": "running", "updated_at": "2026-10-18T11:34:58.172389+00:00"}
Repo card metadata block was not found. Setting CardData to empty.
{
  "dataset": "wandb/RAGTruth-processed",
  "source_split": "test",
  "seed": 42,
  "policy": "Select equal label counts from public test with one case per context; remove all selected contexts from training; skip overlength test inputs intact.",
  "total": 200,
  "skipped": {
    "shared_training_context": 0,
    "duplicate_test_context": 1796,
    "overlength": 121
  },
  "max_length": 1536,
  "model": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "original_training_rows": 10000,
  "training_rows": 10000,
  "training_rows_removed_for_source_overlap": 0,
  "original_train_sha256": "3cba2bca862a57bd3d2d326236db88aec7951d9990d8cfc37e10671c2a1dfcb8",
  "train_sha256": "3cba2bca862a57bd3d2d326236db88aec7951d9990d8cfc37e10671c2a1dfcb8",
  "test_sha256": "080337a2b7d9e930b87c4738bf5dced045b2b73f6671c76ea9d1a2d845cdfe58"


## Important validation

Evaluate the adapter on data that was not used for training. Do not use the frozen project holdout to tune the adapter. Public grounding labels do not teach completeness, instruction following, source conflicts, or project-specific citation policy.